# Benchmarking sign-reading models (issue #56)

Which model reads Linear A signs best? Every candidate is trained with the same recipe (`train.py`) on the same
tablet-level split and scored on the same held-out test tablets.

* **Detectors** find signs (one class, `sign`) - mAP, precision, recall, speed.
* **Classifiers** name each cropped sign - top-1 / top-5.
* **End to end** - the best detector with every classifier, scored on whole tablets by `evaluate.py` with 95% bootstrap CIs.

Use a GPU runtime on Colab (Runtime > Change runtime type). Set `QUICK = True` first to check the pipeline in minutes.

In [ ]:
QUICK = False   # True: 3 epochs per model, numbers are meaningless
REPO = "https://github.com/Rikoleurt/ASE_Group3.git"
BRANCH = "main"

In [ ]:
import os
if not os.path.exists("ASE_Group3"):
    !git clone -b {BRANCH} {REPO}
%cd ASE_Group3
!pip install -q "ultralytics>=8.4,<9.0" matplotlib pandas -r fraction-solver/requirements-data.txt

## 1. Data
Crawls lineara.eu (about 15 min, resumable), then builds the detector and classifier datasets.

In [ ]:
%cd fraction-solver
!python -m la.cli fetch
!python -m la.cli images
!python -m la.cli dataset --labels sign
!python -m la.cli synth
!python -m la.cli crops
%cd ..

## 2. Detectors

In [ ]:
!python benchmark.py detect {'--quick' if QUICK else ''}

## 3. Classifiers

In [ ]:
!python benchmark.py classify {'--quick' if QUICK else ''}

## 4. End to end

In [ ]:
!python benchmark.py end2end

## 5. Results

In [ ]:
!python benchmark.py report
from IPython.display import Image, display
import pandas as pd
for name in ["detect", "classify", "end2end"]:
    p = f"runs/benchmark/{name}.png"
    if os.path.exists(p):
        display(Image(p))

## 6. Conclusion
Results of the full run are summarised in `research/benchmarking/README.md`: yolo26n is the best detector; ConvNeXt-Tiny and EfficientNet-B0 lead the classifiers, but end-to-end differences between classifiers are within the 95% confidence intervals. Replace with your own numbers if you re-run.

In [ ]:
from google.colab import files
!zip -qr benchmark_results.zip runs/benchmark
files.download("benchmark_results.zip")